# SPY 0DTE Credit Spreads — ORB-direction, full sweep with strike aggression

**Window:** last 365 calendar days (≈252 trading days, roughly May 2025 → May 2026).

**Setup:** Opening range = first 30 min. First 1-min close outside OR between 10:00 ET and 13:00 ET (10:00 AM PT) opens a credit spread in the *continuation* direction (break above ORH → bull put; break below ORL → bear call). No Fridays.

**Sweep:** 7 PT × 3 SL × 3 TS × 4 strike offsets (0/1/2/3 OTM) = **252 configs**.

## 1. Setup

In [ ]:
import os, sys
REPO = '/content/iron-condor'
BRANCH = 'claude/spy-options-trading-bot-ri4Ms'
if not os.path.isdir(REPO):
    !git clone --quiet -b {BRANCH} https://github.com/coolin815/iron-condor.git {REPO}
else:
    !cd {REPO} && git pull --quiet
SRC = REPO + '/src'
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.chdir(REPO)
print('OK')

## 2. Paste your Polygon key

In [ ]:
import os, getpass
os.environ['POLYGON_API_KEY'] = getpass.getpass('Polygon API key: ')
print('Key loaded:', len(os.environ['POLYGON_API_KEY']), 'chars')

## 3. Smoke test — one recent day with default params

Confirms the credit spread pipeline works end-to-end (OR detection, signal trigger, strike picker, both-leg option-bar fetching, entry credit calc, walk for exit). Takes ~30 sec.

What to look for in the output:
- `signal_direction`: `bull_put` or `bear_call`
- `short_strike`, `long_strike`, `spread_width`: should match expectations (1-OTM short, 1-wide)
- `entry_credit`: positive number per share (e.g., $0.30)
- `exit_reason`: one of `profit`, `stop`, `time_stop`, or `no_signal`/`no_data`
- `net_pnl`: should be reasonable in size (not zero unless `no_signal`)

In [ ]:
!python -m iron_condor.cli --smoke -v

## 4. Full sweep — 252 configs over a year

First run will fetch new option contracts (4 strike offsets × ~250 days × 2 legs ≈ 2000 new contracts). Expect ~40 min the first time with Polygon throttling. Subsequent reruns ~15 min.

In [ ]:
!python -m iron_condor.cli --days 365 --sweep \
    --short-offset 0 --short-offset 1 --short-offset 2 --short-offset 3

## 5. Top 20 configs

In [ ]:
import pandas as pd
summary = pd.read_csv('results/sweep_summary.csv')
summary.head(20)

## 6. PT × SL heatmaps, broken out by strike offset

In [ ]:
import re
rows = []
for _, r in summary.iterrows():
    m = re.match(r'pt(\d+)\|sl(\d+)\|ts(\d+)\|so([\d.]+)', r['config'])
    if m:
        rows.append({
            'pt': int(m.group(1)),
            'sl': int(m.group(2)),
            'ts': int(m.group(3)),
            'so': float(m.group(4)),
            'return': r['total_return_pct'],
            'win_rate': r['win_rate'],
            'trades': r['n_trades'],
            'end_balance': r['ending_balance'],
        })
grid = pd.DataFrame(rows)
for off in sorted(grid['so'].unique()):
    g = grid[grid['so'] == off]
    print(f'\n=== short-offset = {off} (averaged over time stops) ===')
    print('Return %:')
    print(g.pivot_table(index='pt', columns='sl', values='return', aggfunc='mean').round(1))
    print('Win rate:')
    print(g.pivot_table(index='pt', columns='sl', values='win_rate', aggfunc='mean').round(2))

## 7. Top config breakdown — by direction and exit reason

In [ ]:
trades = pd.read_csv('results/sweep_trades.csv')
top_cfg = summary.iloc[0]['config']
sub = trades[trades['config'] == top_cfg]
taken = sub[sub['exit_reason'].isin(['profit', 'stop', 'time_stop'])]
print(f'Top config: {top_cfg}')
print(f'\nTotal days: {len(sub)}, taken trades: {len(taken)}')
print('\n=== By direction ===')
print(taken.groupby('signal_direction').agg(
    trades=('net_pnl', 'count'),
    win_rate=('net_pnl', lambda s: (s > 0).mean()),
    avg_pnl=('net_pnl', 'mean'),
    total_pnl=('net_pnl', 'sum'),
).round(2))
print('\n=== Exit reasons ===')
print(sub['exit_reason'].value_counts())

## 8. Equity curves — top 5 configs

In [ ]:
import matplotlib.pyplot as plt
top5 = summary.head(5)['config'].tolist()
fig, ax = plt.subplots(figsize=(11, 5))
for cfg in top5:
    sub = trades[trades['config'] == cfg].sort_values('day')
    ax.plot(pd.to_datetime(sub['day']), sub['balance_after'], label=cfg, linewidth=1.5)
ax.axhline(1500, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Date'); ax.set_ylabel('Balance ($)')
ax.set_title('Top-5 credit-spread config equity curves')
ax.legend(fontsize=7); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()